# Write Tags to Database

Write predicted tags from inference results to the RekordBox database.

**IMPORTANT:** Make sure to create a backup before editing the DB. The backup dialog can be found under "File" > "Library" > "Backup Library".

This notebook:
1. Loads tag predictions from the parquet file
2. Writes tags to the database in batches, grouped by song
3. Commits to the database after each batch (finishing all tags for the current song)

In [ ]:
# automatically reload imported modules before executing code
%load_ext autoreload
%autoreload 2

In [ ]:
from nbutils import setup_path, display_polars
import polars as pl
from pathlib import Path

setup_path()

In [ ]:
from pyrekordbox import Rekordbox6Database

db = Rekordbox6Database()

## Configuration

In [ ]:
# Path to the inference results parquet file
INFERENCE_RESULTS_PATH = "../data/full_inference_results_20251018_1440.parquet"

# Batch size: commit to database after this many tags
# Important: we finish all tags for the current song before committing,
# even if it exceeds this number
TAG_COMMIT_BATCH_SIZE = 100

print(f"Configuration:")
print(f"  Input file: {INFERENCE_RESULTS_PATH}")
print(f"  Batch size: {TAG_COMMIT_BATCH_SIZE} tags (per batch, completing current song)")

## Load Predictions

In [ ]:
# Load the predictions
predictions_df = pl.read_parquet(INFERENCE_RESULTS_PATH)

print(f"Loaded predictions:")
print(f"  Total rows: {len(predictions_df)}")
print(f"  Unique songs: {predictions_df.select('song_id').n_unique()}")
print(f"\nColumns: {predictions_df.columns}")
print(f"\nFirst 10 rows:")
display_polars(predictions_df.head(10))

## Verify Tags Exist in Database

Check that all tag IDs in the predictions exist in the database.

In [ ]:
# Get unique tag IDs from predictions
unique_tag_ids = predictions_df.select('tag_id').unique().to_series().to_list()

print(f"Checking {len(unique_tag_ids)} unique tags in database...")

# Check each tag exists
missing_tags = []
for tag_id in unique_tag_ids:
    try:
        tag = db.get_my_tag(ID=str(tag_id))
    except Exception as e:
        missing_tags.append(tag_id)

if missing_tags:
    print(f"\n⚠️  WARNING: {len(missing_tags)} tags not found in database:")
    for tag_id in missing_tags:
        matching_rows = predictions_df.filter(pl.col('tag_id') == tag_id)
        if len(matching_rows) > 0:
            tag_name = matching_rows.select('tag').item(0)
            print(f"  - Tag ID {tag_id}: {tag_name}")
    print("\nPlease create these tags in RekordBox before proceeding.")
else:
    print("✓ All tags exist in database")

## Verify Songs Exist in Database

Check that all song IDs in the predictions exist in the database.

In [ ]:
# Get unique song IDs from predictions
unique_song_ids = predictions_df.select('song_id').unique().to_series().to_list()

print(f"Checking {len(unique_song_ids)} unique songs in database...")

# Check each song exists
missing_songs = []
for song_id in unique_song_ids:
    try:
        content = db.get_content(ID=str(song_id))
    except Exception as e:
        missing_songs.append(song_id)

if missing_songs:
    print(f"\n⚠️  WARNING: {len(missing_songs)} songs not found in database:")
    for song_id in missing_songs[:10]:  # Show first 10
        matching_rows = predictions_df.filter(pl.col('song_id') == song_id)
        if len(matching_rows) > 0:
            song_title = matching_rows.select('song_title').item(0)
            print(f"  - Song ID {song_id}: {song_title}")
    if len(missing_songs) > 10:
        print(f"  ... and {len(missing_songs) - 10} more")
    print("\nThese songs may have been deleted from your RekordBox library.")
else:
    print("✓ All songs exist in database")

## Write Tags to Database

Write tags in batches, ensuring we complete all tags for a song before committing.

In [ ]:
from utils import add_tag
from tqdm.auto import tqdm

# Group predictions by song_id to ensure we process all tags for a song together
song_groups = predictions_df.partition_by('song_id', as_dict=True)

print(f"Writing tags to database...")
print(f"Total songs: {len(song_groups)}")
print(f"Total tags to write: {len(predictions_df)}")
print(f"Batch size: {TAG_COMMIT_BATCH_SIZE} (will finish current song before committing)")
print("\n" + "="*80)

tags_written = 0
songs_processed = 0
batches_committed = 0
errors = []

# Create progress bar
pbar = tqdm(total=len(predictions_df), desc="Writing tags", unit="tags")

for song_id, song_df in song_groups.items():
    song_title = song_df.select('song_title').item(0)
    artist_name = song_df.select('artist_name').item(0)
    
    # Process all tags for this song
    song_tags_written = 0
    
    for row in song_df.iter_rows(named=True):
        tag_id = str(row['tag_id'])
        content_id = str(row['song_id'])
        tag_name = row['tag']
        
        try:
            # Add tag to database (doesn't commit yet)
            add_tag(db, tag_id=tag_id, content_id=content_id)
            tags_written += 1
            song_tags_written += 1
            pbar.update(1)
            
        except ValueError as e:
            # Tag already exists, skip
            if "already associated" in str(e):
                pbar.update(1)
                continue
            else:
                error_msg = f"Error adding tag '{tag_name}' to '{song_title}': {e}"
                errors.append(error_msg)
                pbar.update(1)
                
        except Exception as e:
            error_msg = f"Error adding tag '{tag_name}' to '{song_title}': {e}"
            errors.append(error_msg)
            pbar.update(1)
    
    songs_processed += 1
    
    # Check if we should commit (either reached batch size or this is the last song)
    # We only commit after finishing all tags for the current song
    is_last_song = songs_processed == len(song_groups)
    exceeded_batch = tags_written >= TAG_COMMIT_BATCH_SIZE
    
    if exceeded_batch or is_last_song:
        db.commit()
        batches_committed += 1
        pbar.set_description(f"Writing tags (committed {batches_committed} batches)")
        tags_written = 0  # Reset counter for next batch

pbar.close()

print("\n" + "="*80)
print(f"\n✓ Tag writing complete!")
print(f"  Songs processed: {songs_processed}")
print(f"  Batches committed: {batches_committed}")

if errors:
    print(f"\n⚠️  Encountered {len(errors)} errors:")
    for error in errors[:10]:  # Show first 10 errors
        print(f"  - {error}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more errors")

## Cleanup

Close the database session.

In [ ]:
# Remove any session if it exists
if db.session:
    db.session.close()
    print("Database session closed")

## Summary

Tags have been written to the RekordBox database!

The process:
1. ✓ Loaded predictions from parquet file
2. ✓ Verified all tags and songs exist in database
3. ✓ Wrote tags in batches (grouped by song)
4. ✓ Committed to database after each batch

The tags should now be visible in RekordBox.